In [1]:
import pickle
with open('all_types_1000p_70000t.pkl', 'rb') as file:
    data = pickle.load(file)

data

,coords,trajectory_type
0,"[[-0.5989235263769103, -1.162521574332328, -2....",ATTM
1,"[[-0.08481573554460747, 2.319417467387251, 3.5...",ATTM
2,"[[0.4460439556191179, -0.005415868872246488, -...",ATTM
3,"[[0.005016187430309248, 0.6084933010787497, 0....",ATTM
4,"[[0.018019833626267964, 1.1039231638637528, 1....",ATTM
...,...,...
69995,"[[0.0, -0.7743617637528311, -0.546565095390306...",Brownian Motion
69996,"[[0.0, 1.7572467829976466, 2.0560173343255634,...",Brownian Motion
69997,"[[0.0, 1.17471633827359, 2.036867647381258, 2....",Brownian Motion
69998,"[[0.0, -0.38920630052789046, -0.84620013605425...",Brownian Motion


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import train_test_split

In [3]:
# ---------------------------
# Dataset personalizado
# ---------------------------
class TrajectoryDataset(Dataset):
    def __init__(self, df, label_mapping):
        """
        df: DataFrame con columnas "coords" y "trajectory_type".
        label_mapping: diccionario para convertir string de trajectory_type a int.
        """
        self.df = df.reset_index(drop=True)
        self.label_mapping = label_mapping

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # Cada fila de "coords" es una lista de dos listas (x e y) de 1000 puntos.
        coords = self.df.loc[idx, "coords"]
        # Convertir a array de numpy y luego a tensor.
        # Se espera forma (2, 1000)
        coords = torch.tensor(np.array(coords), dtype=torch.float32)
        label_str = self.df.loc[idx, "trajectory_type"]
        label = self.label_mapping[label_str]
        return coords, label

In [4]:
# ---------------------------
# Codificación Posicional para Transformer
# ---------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # Forma: (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch_size, seq_length, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [5]:
# ---------------------------
# Modelo Transformer para Trayectorias
# ---------------------------
class TrajectoryTransformer(nn.Module):
    def __init__(self, d_model=64, nhead=8, num_encoder_layers=3, num_classes=7, dropout=0.1):
        """
        d_model: Dimensión de embedding.
        nhead: Número de cabezas en la atención multi-head.
        num_encoder_layers: Número de capas del encoder.
        num_classes: Número de clases de salida.
        """
        super().__init__()
        # Proyectamos la entrada 2D (x,y) a una dimensión d_model.
        self.input_proj = nn.Linear(2, d_model)
        
        # Codificación posicional.
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        
        # Bloque encoder transformer.
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=d_model*4, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_encoder_layers)
        
        # Capa de clasificación final: se utiliza un pooling (promedio) sobre la secuencia.
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        """
        x: Tensor de forma (batch_size, canales, seq_length), en este caso canales=2.
        """
        # Transponer para tener (batch_size, seq_length, 2)
        x = x.transpose(1, 2)
        
        # Proyección de entrada: (batch_size, seq_length, d_model)
        x = self.input_proj(x)
        
        # Añadir codificación posicional.
        x = self.pos_encoder(x)
        
        # Transformer requiere la secuencia en la primera dimensión: (seq_length, batch_size, d_model)
        x = x.transpose(0, 1)
        
        # Pasar por el transformer encoder.
        x = self.transformer_encoder(x)
        
        # Hacemos pooling (promedio) en la dimensión de la secuencia.
        x = x.mean(dim=0)  # (batch_size, d_model)
        
        # Clasificación
        x = self.classifier(x)  # (batch_size, num_classes)
        return x

In [6]:
# ---------------------------
# Preparación del Dataset y DataLoader
# ---------------------------
# Supongamos que "data" es tu DataFrame ya cargado.
# data = pd.read_csv('ruta_a_tu_dataset.csv')  # Ajusta según corresponda.

# Si la columna "coords" está en formato string (por ejemplo, json), asegúrate de convertirla al formato correcto (lista de listas).
# Por ejemplo, podrías usar ast.literal_eval:
# import ast
# data['coords'] = data['coords'].apply(ast.literal_eval)

# Crear un mapping para las etiquetas.
unique_labels = data['trajectory_type'].unique()
# Ordenamos para asignar índices de manera consistente.
unique_labels = sorted(unique_labels)
label_mapping = {label: idx for idx, label in enumerate(unique_labels)}
print("Mapping de labels:", label_mapping)

# Dividir en entrenamiento y test (por ejemplo, 80% entrenamiento, 20% test)
train_df, test_df = train_test_split(data, test_size=0.2, stratify=data['trajectory_type'], random_state=42)

train_dataset = TrajectoryDataset(train_df, label_mapping)
test_dataset = TrajectoryDataset(test_df, label_mapping)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

Mapping de labels: {'ATTM': 0, 'Brownian Motion': 1, 'CTRW': 2, 'Fractional Brownian Motion': 3, 'Lévy Walk': 4, 'Random Walk': 5, 'Scaled Brownian Motion': 6}


In [ ]:
# ---------------------------
# Inicialización y Entrenamiento del Modelo
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(label_mapping)

model = TrajectoryTransformer(d_model=64, nhead=8, num_encoder_layers=3, num_classes=num_classes, dropout=0.1)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10  # Ajusta el número de épocas según convenga

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs = inputs.to(device)  # (batch_size, 2, 1000)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)  # (batch_size, num_classes)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_loss = running_loss / total
    train_acc = 100. * correct / total

    # Validación en el set de test.
    model.eval()
    running_loss_test = 0.0
    correct_test = 0
    total_test = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss_test = criterion(outputs, labels)
            running_loss_test += loss_test.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total_test += labels.size(0)
            correct_test += predicted.eq(labels).sum().item()
    
    test_loss = running_loss_test / total_test
    test_acc = 100. * correct_test / total_test

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")

# Una vez entrenado, puedes guardar el modelo o evaluarlo más a fondo.

c:\Users\jmass\Documents\GitHub\TFG_GEINF\env\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
